# Day 56 · Exercise 5: DocStore & Prompt Builder

**What you'll build:** Implement the `DocStore` class (`add`, `get_text`, `list_docs`) and `build_doc_prompt(document_text, question, max_doc_chars)`. These are the last two pieces: a store for uploaded document text and the prompt that sends it to the LLM.

## Setup (provided)

In [ ]:
import io
import re
import secrets
from pathlib import Path
import pypdf

def validate_upload(content: bytes, filename: str,
                    allowed_extensions: list[str], max_bytes: int) -> tuple[bool, str]:
    if len(content) == 0:
        return False, "File is empty"
    if len(content) > max_bytes:
        return False, f"File too large ({len(content)} bytes, max {max_bytes})"
    ext = Path(filename).suffix.lower()
    if ext not in allowed_extensions:
        return False, f"Extension '{ext}' not allowed (allowed: {allowed_extensions})"
    return True, ""

def safe_filename(original: str) -> str:
    name = Path(original).name
    name = re.sub(r'[^\w\-.]', '_', name)
    return f"{secrets.token_hex(4)}_{name}"

def save_upload(content: bytes, filename: str, upload_dir: Path) -> Path:
    upload_dir.mkdir(parents=True, exist_ok=True)
    dest = upload_dir / safe_filename(filename)
    dest.write_bytes(content)
    return dest

def extract_text(content: bytes, content_type: str) -> str:
    if "pdf" in content_type:
        reader = pypdf.PdfReader(io.BytesIO(content))
        return "\n".join(p.extract_text() or "" for p in reader.pages)
    return content.decode("utf-8", errors="replace")


## Your Implementation

In [ ]:
class DocStore:
    """In-memory document store mapping doc_id -> {filename, text}."""

    def __init__(self):
        self._store: dict[str, dict] = {}

    def add(self, filename: str, text: str) -> str:
        """Store a document and return its unique doc_id.

        Args:
            filename: Original filename (stored for display).
            text:     Extracted document text.
        Returns:
            A unique doc_id string (use secrets.token_urlsafe(8)).
        """
        # TODO: generate doc_id = secrets.token_urlsafe(8)
        # store {"filename": filename, "text": text} in self._store[doc_id]
        # return doc_id
        raise NotImplementedError

    def get_text(self, doc_id: str) -> str | None:
        """Return the stored text for doc_id, or None if not found."""
        # TODO: return self._store.get(doc_id, {}).get("text")
        raise NotImplementedError

    def list_docs(self) -> list[dict]:
        """Return [{doc_id, filename}] for all stored documents."""
        # TODO: return [{"doc_id": k, "filename": v["filename"]} for k, v in self._store.items()]
        raise NotImplementedError


def build_doc_prompt(document_text: str, question: str,
                     max_doc_chars: int = 3000) -> str:
    """Build a prompt asking the model a question about the document.

    Args:
        document_text: Extracted document text (may be truncated).
        question:      The user's question.
        max_doc_chars: Maximum characters to include from the document.
    Returns:
        A complete prompt string for the LLM.
    """
    # TODO: truncate document_text to max_doc_chars
    # build a prompt that includes: a system instruction, the truncated document,
    # and the question. Something like:
    # "You are a helpful assistant. Answer based only on the document below.\n\n
    #  DOCUMENT:\n{snippet}\n\nQUESTION: {question}"
    raise NotImplementedError


In [ ]:
class DocStore:
    def __init__(self):
        self._store: dict[str, dict] = {}

    def add(self, filename: str, text: str) -> str:
        doc_id = secrets.token_urlsafe(8)
        self._store[doc_id] = {"filename": filename, "text": text}
        return doc_id

    def get_text(self, doc_id: str) -> str | None:
        return self._store.get(doc_id, {}).get("text")

    def list_docs(self) -> list[dict]:
        return [{"doc_id": k, "filename": v["filename"]} for k, v in self._store.items()]


def build_doc_prompt(document_text: str, question: str,
                     max_doc_chars: int = 3000) -> str:
    snippet = document_text[:max_doc_chars]
    return (
        "You are a helpful assistant. Answer the question based only on the "
        "document below. If the answer is not in the document, say so.\n\n"
        f"DOCUMENT:\n{snippet}\n\n"
        f"QUESTION: {question}"
    )


## Check Your Work

In [ ]:
def _run_checks():
    score = 0
    total = 5

    def _chk(n, ok, msg):
        nonlocal score
        print(f"  {'✅' if ok else '❌'} Check {n}: {msg}")
        if ok:
            score += 1

    try:
        store = DocStore()
        doc_id = store.add("report.txt", "The sky is blue.")
    except NotImplementedError:
        for i in range(1, total + 1):
            print(f"  ❌ Check {i}: DocStore.add not implemented")
        print(f"\nScore: 0 / {total}")
        return

    _chk(1, isinstance(doc_id, str) and len(doc_id) >= 6,
         f"add() returns a non-empty id string (got {doc_id!r})")

    try:
        text = store.get_text(doc_id)
    except NotImplementedError:
        for i in range(2, 4):
            print(f"  ❌ Check {i}: DocStore.get_text not implemented")
        text = None

    _chk(2, text == "The sky is blue.",
         f"get_text returns the stored text (got {text!r})")
    _chk(3, store.get_text("nonexistent") is None,
         "get_text on unknown id returns None")

    # store a second doc
    id2 = store.add("notes.txt", "Rain is wet.")
    try:
        docs = store.list_docs()
    except NotImplementedError:
        print(f"  ❌ Check 4: DocStore.list_docs not implemented")
        docs = None

    _chk(4, isinstance(docs, list) and len(docs) == 2,
         f"list_docs returns 2 entries (got {docs})")

    # build_doc_prompt
    try:
        prompt = build_doc_prompt("The sky is blue.", "What colour is the sky?")
    except NotImplementedError:
        print(f"  ❌ Check 5: build_doc_prompt not implemented")
        print(f"\nScore: {score} / {total}")
        return

    _chk(5, "The sky is blue." in prompt and "What colour" in prompt,
         f"prompt contains document text and question (got {prompt[:80]!r}...)")

    print(f"\nScore: {score} / {total}")
    if score == total:
        print("🎉 Exercise complete!")

_run_checks()


## Bonus Challenge

Modify `DocStore.add` to also store the filename and `len(text)`. Add a `DocStore.summary()` method that returns a list of `{doc_id, filename, char_count}` dicts. This is the metadata a sidebar would show — upload list with document sizes.

## Solution

<details>
<summary>Show solution</summary>

```python
class DocStore:
    def __init__(self):
        self._store: dict[str, dict] = {}

    def add(self, filename: str, text: str) -> str:
        doc_id = secrets.token_urlsafe(8)
        self._store[doc_id] = {"filename": filename, "text": text}
        return doc_id

    def get_text(self, doc_id: str) -> str | None:
        return self._store.get(doc_id, {}).get("text")

    def list_docs(self) -> list[dict]:
        return [{"doc_id": k, "filename": v["filename"]} for k, v in self._store.items()]


def build_doc_prompt(document_text: str, question: str,
                     max_doc_chars: int = 3000) -> str:
    snippet = document_text[:max_doc_chars]
    return (
        "You are a helpful assistant. Answer the question based only on the "
        "document below. If the answer is not in the document, say so.\n\n"
        f"DOCUMENT:\n{snippet}\n\n"
        f"QUESTION: {question}"
    )
```

**Why this works:** `DocStore` uses a dict keyed by a random `token_urlsafe(8)` id.
The id is generated at store-time, not at init-time, so each document gets a unique
handle. `get_text` returns `None` for unknown ids — the caller raises 404.
`build_doc_prompt` truncates the document with a slice before embedding it, so the
LLM never receives more than `max_doc_chars` characters of context. The prompt
explicitly tells the model to answer only from the document — this is the core of
a retrieval-augmented Q&A system.

</details>